# Deep Image Prior : Foundation & Classical Baselines

## 📋 Project Overview

This notebook implements the foundational components for our Deep Image Prior project:

1. **Dataset Exploration** - BSDS300 test images
2. **Image Degradation** - Implementing $y = Ax + \eta$
3. **Classical Baselines** - Wiener Filter & BM3D
4. **Evaluation Metrics** - PSNR & SSIM comparison table

### The Forward Model

$$y = Ax + \eta$$

Where:
- $x$: Clean ground-truth image
- $A$: Degradation operator (Identity for denoising, Blur kernel for deconvolution)
- $\eta$: Additive Gaussian noise $\sim \mathcal{N}(0, \sigma^2)$
- $y$: Observed degraded image

## 1. Setup & Imports

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, str(Path.cwd() / 'src'))

# Import our modules
from metrics import psnr, ssim, evaluate_image, print_metrics
from degrade import (load_image, save_image, add_gaussian_noise, 
                     gaussian_kernel, apply_blur, 
                     degrade_for_denoising, degrade_for_deconvolution)
from baselines import (wiener_deconvolution, bm3d_denoise, 
                       total_variation_denoise, restore_image, HAS_BM3D)

# Display settings
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['figure.dpi'] = 100

print("✓ All modules imported successfully!")
print(f"✓ BM3D available: {HAS_BM3D}")

## 2. Dataset Exploration

Select 5-10 test images from BSDS300 for consistent evaluation.

In [ ]:
# Path to dataset
DATASET_PATH = Path('Dataset/BSDS300/images/test')

# List available images
image_files = sorted(list(DATASET_PATH.glob('*.jpg')))
print(f"Found {len(image_files)} test images in BSDS300")

# Select 10 images for our evaluation set
# Choosing diverse images with different characteristics
SELECTED_IMAGES = [
    '101085.jpg',  # Natural scene
    '102061.jpg',  # Texture
    '108005.jpg',  # Portrait-like
    '126007.jpg',  # Architecture
    '167062.jpg',  # Fine details
    '175032.jpg',  # Landscape
    '208001.jpg',  # Mixed content
    '253027.jpg',  # Complex texture
    '291000.jpg',  # High contrast
    '3096.jpg',    # Natural scene
]

# Verify all selected images exist
test_images = []
for img_name in SELECTED_IMAGES:
    img_path = DATASET_PATH / img_name
    if img_path.exists():
        test_images.append(img_path)
        print(f"  ✓ {img_name}")
    else:
        print(f"  ✗ {img_name} not found")

print(f"\nSelected {len(test_images)} images for evaluation")

In [ ]:
# Display the selected test images
fig, axes = plt.subplots(2, 5, figsize=(15, 7))
axes = axes.flatten()

for i, img_path in enumerate(test_images[:10]):
    img = load_image(img_path, normalize=True, grayscale=False)
    axes[i].imshow(img)
    axes[i].set_title(img_path.name, fontsize=9)
    axes[i].axis('off')

plt.suptitle('BSDS300 Test Set - Selected Evaluation Images', fontsize=14)
plt.tight_layout()
plt.show()

# Print image statistics
sample_img = load_image(test_images[0])
print(f"\nImage Statistics:")
print(f"  Shape: {sample_img.shape}")
print(f"  Dtype: {sample_img.dtype}")
print(f"  Range: [{sample_img.min():.3f}, {sample_img.max():.3f}]")

## 3. Image Degradation: $y = Ax + \eta$

### 3.1 Denoising Scenario
$A = I$ (Identity matrix), Gaussian noise $\sigma = 0.1$ or $0.2$

In [ ]:
# Load a sample image
sample_path = test_images[0]
clean = load_image(sample_path, normalize=True, grayscale=False)

# Create noisy versions with different sigma values
sigma_values = [0.05, 0.1, 0.2, 0.3]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

# Original
axes[0].imshow(clean)
axes[0].set_title('Clean Original', fontsize=11)
axes[0].axis('off')

# Noisy versions
for i, sigma in enumerate(sigma_values):
    noisy, _ = degrade_for_denoising(clean, sigma=sigma, seed=42)
    noisy_psnr = psnr(clean, noisy)
    
    axes[i+1].imshow(np.clip(noisy, 0, 1))
    axes[i+1].set_title(f'σ = {sigma}\nPSNR: {noisy_psnr:.1f} dB', fontsize=11)
    axes[i+1].axis('off')

plt.suptitle('Denoising Scenario: y = x + η (Gaussian Noise)', fontsize=14)
plt.tight_layout()
plt.show()

### 3.2 Deconvolution Scenario
$A$ is a blur kernel (Gaussian, Motion, etc.), noise $\sigma = 0.01$

In [ ]:
from degrade import gaussian_kernel, motion_blur_kernel, box_kernel, disk_kernel

# Show different blur kernels
kernels = {
    'Gaussian 7×7': gaussian_kernel(7, 1.5),
    'Gaussian 13×13': gaussian_kernel(13, 2.5),
    'Motion 15×15': motion_blur_kernel(15, 45),
    'Box 7×7': box_kernel(7),
}

fig, axes = plt.subplots(2, 4, figsize=(14, 8))

for i, (name, kernel) in enumerate(kernels.items()):
    # Show kernel
    axes[0, i].imshow(kernel, cmap='hot')
    axes[0, i].set_title(f'{name}', fontsize=10)
    axes[0, i].axis('off')
    
    # Apply blur + noise
    blurred = apply_blur(clean, kernel)
    blurry_noisy = add_gaussian_noise(blurred, sigma=0.01, seed=42)
    blurry_psnr = psnr(clean, blurry_noisy)
    
    axes[1, i].imshow(np.clip(blurry_noisy, 0, 1))
    axes[1, i].set_title(f'Blurred + Noise\nPSNR: {blurry_psnr:.1f} dB', fontsize=10)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Blur Kernel', fontsize=11)
axes[1, 0].set_ylabel('Degraded Image', fontsize=11)

plt.suptitle('Deconvolution Scenario: y = A*x + η (Blur + Noise)', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Classical Baselines Evaluation

### 4.1 Wiener Filter & BM3D on Denoising Task

In [ ]:
# Test on a single image first
NOISE_SIGMA = 0.1  # Standard denoising scenario

# Create degraded image
noisy, params = degrade_for_denoising(clean, sigma=NOISE_SIGMA, seed=42)

# Apply baselines
wiener_result = wiener_deconvolution(noisy, kernel=None)  # Denoising mode
tv_result = total_variation_denoise(noisy, weight=0.15)

if HAS_BM3D:
    bm3d_result = bm3d_denoise(noisy, sigma=NOISE_SIGMA)
else:
    print("⚠ BM3D not installed, using TV as placeholder")
    bm3d_result = tv_result

# Compute metrics
results = {
    'Noisy': evaluate_image(clean, noisy),
    'Wiener': evaluate_image(clean, wiener_result),
    'TV': evaluate_image(clean, tv_result),
    'BM3D': evaluate_image(clean, bm3d_result) if HAS_BM3D else {'psnr': 0, 'ssim': 0},
}

print(f"\nDenoising Results (σ = {NOISE_SIGMA}):")
print("="*50)
for method, metrics in results.items():
    print(f"  {method:8}: PSNR = {metrics['psnr']:5.2f} dB, SSIM = {metrics['ssim']:.4f}")
print("="*50)

In [ ]:
# Visual comparison
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

images = [
    (clean, 'Ground Truth'),
    (noisy, f"Noisy (σ={NOISE_SIGMA})\nPSNR: {results['Noisy']['psnr']:.2f} dB"),
    (wiener_result, f"Wiener Filter\nPSNR: {results['Wiener']['psnr']:.2f} dB"),
    (tv_result, f"Total Variation\nPSNR: {results['TV']['psnr']:.2f} dB"),
    (bm3d_result, f"BM3D\nPSNR: {results['BM3D']['psnr']:.2f} dB"),
]

for ax, (img, title) in zip(axes.flatten(), images):
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(title, fontsize=11)
    ax.axis('off')

axes[1, 2].axis('off')  # Empty subplot

plt.suptitle('Classical Denoising Methods Comparison', fontsize=14)
plt.tight_layout()
plt.show()

### 4.2 Zoomed Comparison (Detail Analysis)

In [ ]:
# Zoom into a region to see fine details
# Crop a 128x128 region from the image
y1, y2, x1, x2 = 100, 228, 150, 278

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

crops = [
    (clean[y1:y2, x1:x2], 'Ground Truth'),
    (noisy[y1:y2, x1:x2], 'Noisy'),
    (wiener_result[y1:y2, x1:x2], 'Wiener'),
    (tv_result[y1:y2, x1:x2], 'TV'),
    (bm3d_result[y1:y2, x1:x2], 'BM3D'),
]

for ax, (crop, title) in zip(axes, crops):
    ax.imshow(np.clip(crop, 0, 1))
    ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.suptitle('Zoomed Detail Comparison (128×128 crop)', fontsize=14)
plt.tight_layout()
plt.show()

print("\nObservations:")
print("- Wiener: May show ringing artifacts, works best with known noise statistics")
print("- TV: Preserves edges but can create 'cartoon-like' appearance")
print("- BM3D: Best texture preservation, but can over-smooth fine details")

## 5. Full Benchmark: PSNR Table

Run all baselines on our 10 selected test images.

In [ ]:
# Configuration
NOISE_SIGMA = 0.1
SEED = 42

# Storage for results
benchmark_results = {
    'image_id': [],
    'noisy_psnr': [],
    'noisy_ssim': [],
    'wiener_psnr': [],
    'wiener_ssim': [],
    'tv_psnr': [],
    'tv_ssim': [],
    'bm3d_psnr': [],
    'bm3d_ssim': [],
}

print(f"Running benchmark on {len(test_images)} images (σ = {NOISE_SIGMA})...\n")

for i, img_path in enumerate(test_images):
    img_id = img_path.stem
    print(f"Processing [{i+1}/{len(test_images)}]: {img_id}")
    
    # Load and degrade
    clean = load_image(img_path, normalize=True, grayscale=False)
    noisy, _ = degrade_for_denoising(clean, sigma=NOISE_SIGMA, seed=SEED+i)
    
    # Apply baselines
    wiener_result = wiener_deconvolution(noisy, kernel=None)
    tv_result = total_variation_denoise(noisy, weight=0.15)
    
    if HAS_BM3D:
        bm3d_result = bm3d_denoise(noisy, sigma=NOISE_SIGMA)
    else:
        bm3d_result = tv_result  # Fallback
    
    # Compute metrics
    benchmark_results['image_id'].append(img_id)
    
    noisy_m = evaluate_image(clean, noisy)
    benchmark_results['noisy_psnr'].append(noisy_m['psnr'])
    benchmark_results['noisy_ssim'].append(noisy_m['ssim'])
    
    wiener_m = evaluate_image(clean, wiener_result)
    benchmark_results['wiener_psnr'].append(wiener_m['psnr'])
    benchmark_results['wiener_ssim'].append(wiener_m['ssim'])
    
    tv_m = evaluate_image(clean, tv_result)
    benchmark_results['tv_psnr'].append(tv_m['psnr'])
    benchmark_results['tv_ssim'].append(tv_m['ssim'])
    
    bm3d_m = evaluate_image(clean, bm3d_result)
    benchmark_results['bm3d_psnr'].append(bm3d_m['psnr'])
    benchmark_results['bm3d_ssim'].append(bm3d_m['ssim'])

print("\n✓ Benchmark complete!")

In [ ]:
# Display PSNR table
print("\n" + "="*85)
print(f"{'DENOISING BENCHMARK RESULTS (σ = ' + str(NOISE_SIGMA) + ')':^85}")
print("="*85)
print(f"{'Image ID':<12} | {'Noisy':>10} | {'Wiener':>10} | {'TV':>10} | {'BM3D':>10} |")
print("-"*85)

for i in range(len(benchmark_results['image_id'])):
    print(f"{benchmark_results['image_id'][i]:<12} | "
          f"{benchmark_results['noisy_psnr'][i]:>8.2f} dB | "
          f"{benchmark_results['wiener_psnr'][i]:>8.2f} dB | "
          f"{benchmark_results['tv_psnr'][i]:>8.2f} dB | "
          f"{benchmark_results['bm3d_psnr'][i]:>8.2f} dB |")

print("-"*85)

# Averages
avg_noisy = np.mean(benchmark_results['noisy_psnr'])
avg_wiener = np.mean(benchmark_results['wiener_psnr'])
avg_tv = np.mean(benchmark_results['tv_psnr'])
avg_bm3d = np.mean(benchmark_results['bm3d_psnr'])

print(f"{'AVERAGE':<12} | {avg_noisy:>8.2f} dB | {avg_wiener:>8.2f} dB | "
      f"{avg_tv:>8.2f} dB | {avg_bm3d:>8.2f} dB |")
print("="*85)

In [ ]:
# Display SSIM table
print("\n" + "="*75)
print(f"{'SSIM RESULTS':^75}")
print("="*75)
print(f"{'Image ID':<12} | {'Noisy':>10} | {'Wiener':>10} | {'TV':>10} | {'BM3D':>10} |")
print("-"*75)

for i in range(len(benchmark_results['image_id'])):
    print(f"{benchmark_results['image_id'][i]:<12} | "
          f"{benchmark_results['noisy_ssim'][i]:>10.4f} | "
          f"{benchmark_results['wiener_ssim'][i]:>10.4f} | "
          f"{benchmark_results['tv_ssim'][i]:>10.4f} | "
          f"{benchmark_results['bm3d_ssim'][i]:>10.4f} |")

print("-"*75)

# Averages
avg_noisy_ssim = np.mean(benchmark_results['noisy_ssim'])
avg_wiener_ssim = np.mean(benchmark_results['wiener_ssim'])
avg_tv_ssim = np.mean(benchmark_results['tv_ssim'])
avg_bm3d_ssim = np.mean(benchmark_results['bm3d_ssim'])

print(f"{'AVERAGE':<12} | {avg_noisy_ssim:>10.4f} | {avg_wiener_ssim:>10.4f} | "
      f"{avg_tv_ssim:>10.4f} | {avg_bm3d_ssim:>10.4f} |")
print("="*75)

## 6. Visualization: Bar Chart Comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Bar chart comparison
methods = ['Noisy', 'Wiener', 'TV', 'BM3D']
psnr_means = [avg_noisy, avg_wiener, avg_tv, avg_bm3d]
ssim_means = [avg_noisy_ssim, avg_wiener_ssim, avg_tv_ssim, avg_bm3d_ssim]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# PSNR bars
colors = ['#ff6b6b', '#4ecdc4', '#45b7d1', '#96ceb4']
bars1 = ax1.bar(methods, psnr_means, color=colors, edgecolor='black', linewidth=1.2)
ax1.set_ylabel('PSNR (dB)', fontsize=12)
ax1.set_title('Average PSNR by Method', fontsize=13)
ax1.set_ylim(0, max(psnr_means) * 1.15)

# Add value labels
for bar, val in zip(bars1, psnr_means):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{val:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# SSIM bars
bars2 = ax2.bar(methods, ssim_means, color=colors, edgecolor='black', linewidth=1.2)
ax2.set_ylabel('SSIM', fontsize=12)
ax2.set_title('Average SSIM by Method', fontsize=13)
ax2.set_ylim(0, 1.1)

# Add value labels
for bar, val in zip(bars2, ssim_means):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.suptitle(f'Classical Baseline Performance (σ = {NOISE_SIGMA})', fontsize=14)
plt.tight_layout()
plt.savefig('results/baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Plot saved to results/baseline_comparison.png")

## 7. Create Comparison Montage

Generate the "Money Plot": [Original | Noisy | Wiener | BM3D]

In [ ]:
# Create montage for 4 selected images
selected_for_montage = test_images[:4]

fig, axes = plt.subplots(4, 5, figsize=(18, 14))

for row, img_path in enumerate(selected_for_montage):
    # Load and degrade
    clean = load_image(img_path, normalize=True, grayscale=False)
    noisy, _ = degrade_for_denoising(clean, sigma=NOISE_SIGMA, seed=SEED+row)
    
    # Apply methods
    wiener_result = wiener_deconvolution(noisy, kernel=None)
    tv_result = total_variation_denoise(noisy, weight=0.15)
    bm3d_result = bm3d_denoise(noisy, sigma=NOISE_SIGMA) if HAS_BM3D else tv_result
    
    # Calculate PSNR
    noisy_psnr = psnr(clean, noisy)
    wiener_psnr = psnr(clean, wiener_result)
    tv_psnr = psnr(clean, tv_result)
    bm3d_psnr = psnr(clean, bm3d_result)
    
    # Display
    images_row = [
        (clean, 'Original'),
        (noisy, f'Noisy\n{noisy_psnr:.1f} dB'),
        (wiener_result, f'Wiener\n{wiener_psnr:.1f} dB'),
        (tv_result, f'TV\n{tv_psnr:.1f} dB'),
        (bm3d_result, f'BM3D\n{bm3d_psnr:.1f} dB'),
    ]
    
    for col, (img, title) in enumerate(images_row):
        axes[row, col].imshow(np.clip(img, 0, 1))
        if row == 0:
            axes[row, col].set_title(title, fontsize=11)
        else:
            # Just show PSNR for other rows
            axes[row, col].set_title(title.split('\n')[-1] if '\n' in title else '', fontsize=10)
        axes[row, col].axis('off')
    
    # Add image name on the left
    axes[row, 0].set_ylabel(img_path.stem, fontsize=10, rotation=0, labelpad=50)

plt.suptitle(f'Comparison Montage: Classical Denoising Methods (σ = {NOISE_SIGMA})', fontsize=14)
plt.tight_layout()
plt.savefig('results/comparison_montage.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Montage saved to results/comparison_montage.png")

## 8. Save Results to CSV

In [ ]:
import csv
from pathlib import Path

# Create results directory
Path('results').mkdir(exist_ok=True)

# Save PSNR/SSIM results
csv_path = 'results/baseline_benchmark.csv'

with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    
    # Header
    writer.writerow(['Image ID', 'Noisy PSNR', 'Wiener PSNR', 'TV PSNR', 'BM3D PSNR',
                     'Noisy SSIM', 'Wiener SSIM', 'TV SSIM', 'BM3D SSIM'])
    
    # Data rows
    for i in range(len(benchmark_results['image_id'])):
        writer.writerow([
            benchmark_results['image_id'][i],
            f"{benchmark_results['noisy_psnr'][i]:.2f}",
            f"{benchmark_results['wiener_psnr'][i]:.2f}",
            f"{benchmark_results['tv_psnr'][i]:.2f}",
            f"{benchmark_results['bm3d_psnr'][i]:.2f}",
            f"{benchmark_results['noisy_ssim'][i]:.4f}",
            f"{benchmark_results['wiener_ssim'][i]:.4f}",
            f"{benchmark_results['tv_ssim'][i]:.4f}",
            f"{benchmark_results['bm3d_ssim'][i]:.4f}",
        ])
    
    # Averages
    writer.writerow([
        'AVERAGE',
        f"{avg_noisy:.2f}",
        f"{avg_wiener:.2f}",
        f"{avg_tv:.2f}",
        f"{avg_bm3d:.2f}",
        f"{avg_noisy_ssim:.4f}",
        f"{avg_wiener_ssim:.4f}",
        f"{avg_tv_ssim:.4f}",
        f"{avg_bm3d_ssim:.4f}",
    ])

print(f"✓ Results saved to {csv_path}")

## 9. Summary & Next Steps

### What We Accomplished (Weeks 1-4):

1. ✅ **Dataset Preparation**: Selected 10 test images from BSDS300
2. ✅ **Degradation Pipeline**: Implemented $y = Ax + \eta$ for denoising and deconvolution
3. ✅ **Classical Baselines**: Wiener Filter, Total Variation, BM3D
4. ✅ **Evaluation Metrics**: PSNR and SSIM computation
5. ✅ **Benchmark Table**: Quantitative comparison across all test images

### Key Observations:

- **BM3D** typically achieves the highest PSNR (often 27-30+ dB)
- **TV Denoising** preserves edges but may over-smooth textures
- **Wiener Filter** works best when noise statistics are known

### Next Steps (Month 2):

1. **Build U-Net Architecture** (Week 5-6)
   - Skip connections for multi-scale features
   - Input: Random noise tensor
   - Target: Degraded image

2. **Implement DIP Optimization** (Week 7-8)
   - Adam optimizer
   - Early stopping to find "sweet spot"
   - Create restoration GIF

### The DIP Goal:
> Beat BM3D's PSNR using an **untrained** neural network that learns from a **single image**!

In [ ]:
print("\n" + "="*60)
print("MONTH 1 COMPLETE: Foundation & Classical Baselines")
print("="*60)
print(f"\n📊 Benchmark Summary (σ = {NOISE_SIGMA}):")
print(f"   - Noisy Input:  {avg_noisy:.2f} dB PSNR")
print(f"   - Wiener:       {avg_wiener:.2f} dB PSNR (+{avg_wiener-avg_noisy:.2f} dB)")
print(f"   - TV:           {avg_tv:.2f} dB PSNR (+{avg_tv-avg_noisy:.2f} dB)")
print(f"   - BM3D:         {avg_bm3d:.2f} dB PSNR (+{avg_bm3d-avg_noisy:.2f} dB)")
print(f"\n🎯 Target for DIP: Beat {avg_bm3d:.2f} dB!")
print("="*60)